In [0]:
# import libraries

from pyspark.sql import SparkSession, functions as F

from pyspark.sql.functions import (
    explode, desc,  row_number, col, try_divide, year, try_to_date, count, 
    countDistinct
)

from pyspark.sql.window import Window

# import pandas as pd


In [0]:
# curl https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json
# record a file in local wget -O steam_game_output.json https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json

In [0]:
# from pyspark.sql import SparkSession

filepath = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

df = spark.read.format('json').load(filepath)

In [0]:
# Number of elements in dataframe
print(f"Entries number : {df.count()}")


Entries number : 55691


The dataset is to big for json_normalize method.

In [0]:
type(df)

pyspark.sql.connect.dataframe.DataFrame

In [0]:
df.take(1)

[Row(data=Row(appid=10, categories=['Multi-player', 'Valve Anti-Cheat enabled', 'Online PvP', 'Shared/Split Screen PvP', 'PvP'], ccu=13990, developer='Valve', discount='0', genre='Action', header_image='https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513', initialprice='999', languages='English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean', name='Counter-Strike', negative=5199, owners='10,000,000 .. 20,000,000', platforms=Row(linux=True, mac=True, windows=True), positive=201215, price='999', publisher='Valve', release_date='2000/11/1', required_age='0', short_description="Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.", tags=Row(1980s=266, 1990's=1191, 2.5

In [0]:
df.printSchema()

root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nullable = true)
 |    |-

We observe 23 different values : data is a only a node, categories, platforms and tags contain nested informations. Let's flatened the dataframe to range datas at same level. 

In [0]:
from pyspark.sql.functions import concat_ws

flat_df = df.select("id","data.*", concat_ws(", ", "data.categories").alias("category"))
flat_df.limit(1).display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,"Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP"


In [0]:
# from pyspark.sql import functions as F

flat_df = flat_df.withColumn(
    "platform",
    F.concat_ws(', ', *[F.when(F.col(f"platforms.{f}"), F.lit(f)) for f in ['linux', 'mac', 'windows']])
)
flat_df.limit(1).display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category,platform
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,"Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP","linux, mac, windows"


In [0]:
flat_df = flat_df.withColumn("tag", F.to_json(F.col("tags")))
flat_df.limit(1).display()

id,appid,categories,ccu,developer,discount,genre,header_image,initialprice,languages,name,negative,owners,platforms,positive,price,publisher,release_date,required_age,short_description,tags,type,website,category,platform,tag
10,10,"List(Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP)",13990,Valve,0,Action,https://cdn.akamai.steamstatic.com/steam/apps/10/header.jpg?t=1666823513,999,"English, French, German, Italian, Spanish - Spain, Simplified Chinese, Traditional Chinese, Korean",Counter-Strike,5199,"10,000,000 .. 20,000,000","List(true, true, true)",201215,999,Valve,2000/11/1,0,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.,"List(266, 1191, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 5426, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 227, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 2784, null, null, null, null, null, null, null, null, null, null, null, null, 1607, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 4831, null, null, null, null, null, null, null, null, null, 1707, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 632, null, null, null, null, null, null, null, null, null, null, null, 3392, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 131, null, null, 769, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 881, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 289, null, null, null, 3353, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 614, null, null, null, null, null, null, 304, null, null, null, 1344, null, null, 1864, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, 1192)",game,,"Multi-player, Valve Anti-Cheat enabled, Online PvP, Shared/Split Screen PvP, PvP","linux, mac, windows","{""1980s"":266,""1990's"":1191,""Action"":5426,""Assassin"":227,""Classic"":2784,""Competitive"":1607,""FPS"":4831,""First-Person"":1707,""Military"":632,""Multiplayer"":3392,""Nostalgia"":131,""Old School"":769,""PvP"":881,""Score Attack"":289,""Shooter"":3353,""Strategy"":614,""Survival"":304,""Tactical""

In [0]:
flat_df = flat_df.drop( "categories", "platforms", "tags")

In [0]:
flat_df.printSchema()

root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- ccu: long (nullable = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- header_image: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- type: string (nullable = true)
 |-- website: string (nullable = true)
 |-- category: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)



In [0]:
# flat_df.write.csv("df.csv", header=True)

Export a csv with the .write method requires missing write right on working directory in databricks free edition. So, let's use pandas method .tocsv()

In [0]:
# import pandas as pd

# flat_df.toPandas().to_csv("df.csv", index=False)

# Explorary dataset analysis

## 1. Explore dataset

In [0]:
len(flat_df.columns)

23

In [0]:
flat_df.count()

55691

The dataframe has 23 columns and 55691 rows.

Now, let's query on the dataframe. Spark lets us run classic SQL queries on your tables, however, using classic SQL in Spark requires you to load the data in memory before running any query. We will use the .createOrReplaceTempView Spark DataFrame method in order to load the data in memory under a certain table name, we will then be able to run SQL queries on it.

In [0]:
flat_df.createOrReplaceTempView('temp_table') # Creates a temporary view table in memory called temp_table
result = spark.sql("select * from temp_table limit 1")
result.show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,00

The .sql method lets you write queries in SQL while benefiting from the distributed computing advantages of Spark.

In [0]:
result = spark.sql("SELECT * FROM temp_table LIMIT 1") # filters elements from temp_table 
# show everything
result.show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,00

We observe 
- ccu nombre de joueurs qui jouaient simulatanément au moment où le dataset a été construit
- discount en pourcentage
- prix stocké en centimes de dollars
- owners indique un intervalle pour évaluer le nombre d'acheteurs
- categories la liste varie selon le jeu. Elles permettent aux acheteurs de classer les jeux et fonctionnent comme les étagères d'une bibliothèque.
- duplicated rows ?

some problems :
- unused spaces : sort columns to verify wether it's a display problem or requests problem 
- different spelling : lowercase string columns
- special alphabetic characters on string columns developers '---', "flyingcubicle, - ", "((no-end-parens studio", "revday studio", "+7 software" , "+mpact games, llc."  
- empty entries : '' on website and platform -> verify other columns
- unused columns : id and appid are identifyers, we use name and can delete them, website, header_image 
- des colonnes numériques au format string -> convertir price, initialprice et discount au format long
- release_date string for datetime

We must verify with steam team before treat these data.
- special alphabetic characters : "," is a separator that we can use to count entries on developers, genre, languages. 
Are " ", None, ., CD PROJEKT RED, [2.21] real publishers and '---', "flyingcubicle, - ", "((no-end-parens studio", "revday studio", "+7 software" , "+mpact games, llc." real developers ?
Are "(none)", " ", "-" real publishers ?
- asiatic characters. We need information even if the langage changes.

Let's sample and observe values on selected columns.

In [0]:
flat_df.select('developer', 'publisher', 'owners', 'ccu', 'website', 'header_image').sample(fraction=0.0001).distinct().show(truncate=False)

+------------------------+----------------------+--------------------+---+------------------------------+-----------------------------------------------------------------------------+
|developer               |publisher             |owners              |ccu|website                       |header_image                                                                 |
+------------------------+----------------------+--------------------+---+------------------------------+-----------------------------------------------------------------------------+
|KiKi                    |KiKi                  |0 .. 20,000         |0  |                              |https://cdn.akamai.steamstatic.com/steam/apps/1079340/header.jpg?t=1558501720|
|Prestige Games          |Prestige Games        |0 .. 20,000         |0  |                              |https://cdn.akamai.steamstatic.com/steam/apps/1625840/header.jpg?t=1646069423|
|Eutopia Studios         |Eutopia Studios       |0 .. 20,000         |0  |      

In [0]:
spark.sql("select * from temp_table limit 5").show()

+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|                    short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------

In [0]:
result = spark.sql("select distinct type from temp_table")
result.show()

+--------+
|    type|
+--------+
|hardware|
|    game|
+--------+



In [0]:
result = spark.sql("select * from temp_table where type = 'hardware'")
result.show(truncate=False)

+------+------+---+---------+--------+-----+----------------------------------------------------------------------------+------------+---------+----------+--------+--------------------+--------+-----+-----------+------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+-----------------------------------------+---------------------------------------------+-------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id    |appid |ccu|developer|discount|genre|header_image                                                                |initialprice|languages|name      

Steam link est un outil édité par Anima Locus qui permet d'accéder à un jeu par lien sur le téléphone, une tablette, un autre PC et même d'accéder à un jeu hébergé sur le pc d'un ami. Ce n'est pas un jeu en soi. Il convient de supprimer la colonne type.

In [0]:
result = spark.sql("select distinct name from temp_table where name like'Counter-Strike%'")
result.show(truncate=False)

+--------------------------------+
|name                            |
+--------------------------------+
|Counter-Strike Nexon: Studio    |
|Counter-Strike: Global Offensive|
|Counter-Strike: Source          |
|Counter-Strike: Condition Zero  |
|Counter-Strike                  |
+--------------------------------+



In [0]:
from pyspark.sql.functions import col

string_cols = [field.name for field in flat_df.schema.fields if field.dataType.simpleString() == 'string']

for c in string_cols:
    count = flat_df.filter(col(c) == '').count()
    if count > 0:
        print(f"{c}: {count}")

developer: 127
genre: 161
languages: 11
publisher: 132
release_date: 99
short_description: 37
website: 25217
category: 970


In [0]:
result = spark.sql("select * from temp_table where release_date = ''")
result.show()

+-------+-------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|ccu|           developer|discount|               genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|           platform|                 tag|
+-------+-------+---+--------------------+--------+--------------------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+-------------

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

all_cols = flat_df.columns
string_cols = [f.name for f in flat_df.schema.fields if f.dataType.simpleString() == 'string']

flat_df.select([spark_sum((col(c).isNull() | ((col(c) == '') if c in string_cols else False)).cast('int')).alias(c) for c in all_cols]).show()

+---+-----+---+---------+--------+-----+------------+------------+---------+----+--------+------+--------+-----+---------+------------+------------+-----------------+----+-------+--------+--------+---+
| id|appid|ccu|developer|discount|genre|header_image|initialprice|languages|name|negative|owners|positive|price|publisher|release_date|required_age|short_description|type|website|category|platform|tag|
+---+-----+---+---------+--------+-----+------------+------------+---------+----+--------+------+--------+-----+---------+------------+------------+-----------------+----+-------+--------+--------+---+
|  0|    0|  0|      127|       0|  161|           0|           0|       11|   0|       0|     0|       0|    0|      132|          99|           0|               37|   0|  25217|     970|       0|  0|
+---+-----+---+---------+--------+-----+------------+------------+---------+----+--------+------+--------+-----+---------+------------+------------+-----------------+----+-------+--------+----

There are 55691 rows. Website misses on one half rows. Other missing values should be converted.


In [0]:
flat_df.count() == flat_df.dropDuplicates().count()

True

In [0]:
(flat_df.count() - flat_df.dropDuplicates().count())

0

There is no duplicated value.

## 2. Clean data

Let's record clean_data under a new data_frame and create a new temp view.

In [0]:
clean_df = flat_df
clean_df.createOrReplaceTempView('table')

### a) treatments

Let's clean data. First, delete unused spaces with trim method.

In [0]:
clean_df = spark.sql("SELECT * FROM table LIMIT 1")
clean_df.show()

+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| id|appid|  ccu|developer|discount| genre|        header_image|initialprice|           languages|          name|negative|              owners|positive|price|publisher|release_date|required_age|   short_description|type|website|            category|           platform|                 tag|
+---+-----+-----+---------+--------+------+--------------------+------------+--------------------+--------------+--------+--------------------+--------+-----+---------+------------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+
| 10|   10|13990|    Valve|       0|Action|https://cdn.akama...|         999|English, French, ...|Counter-Strike|    5199|10,00

In [0]:
from pyspark.sql.functions import trim

clean_df = spark.sql("SELECT * FROM table") \
    .select([trim(col(c)).alias(c) for c in spark.sql("SELECT * FROM table").columns])
clean_df.show()


+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+--------------------+--------+-----+----------------------------+------------+------------+-------------------------------------+----+--------------------+--------------------+-------------------+--------------------+
|     id|  appid|  ccu|                   developer|discount|               genre|        header_image|initialprice|           languages|                                name|negative|              owners|positive|price|                   publisher|release_date|required_age|                    short_description|type|             website|            category|           platform|                 tag|
+-------+-------+-----+----------------------------+--------+--------------------+--------------------+------------+--------------------+------------------------------------+--------+---------------

Then, convert numerical columns ccu, price, initialprice and discount from string to long.

In [0]:
from pyspark.sql.functions import col

string_cols = ['ccu', 'price', 'initialprice', 'discount']

for c in string_cols:
    clean_df = clean_df.withColumn(c, col(c).cast('long'))


Then, convert release_date from string to datetime

to_date() ne passe pas en raison de valeurs non conformes. Comptez le nombre de segments (/) pour chaque valeur distincte de 'release_date', afin de voir toutes les structures présentes (3 segments, 2 segments, etc.).

In [0]:
from pyspark.sql.functions import col, split, size

clean_df.select('release_date', size(split(col('release_date'), '/')).alias('partition_nb')) \
    .distinct() \
    .groupBy('partition_nb') \
    .count() \
    .show()

+------------+-----+
|partition_nb|count|
+------------+-----+
|           3| 3937|
|           2|   73|
|           1|    1|
+------------+-----+



Résultat clair: Trois structures existent dans 'release_date':

3 segments (3937 lignes): format complet 'yyyy/M/d'
2 segments (73 lignes): année/mois seulement, ex '2019/01'
1 segment (1 ligne): probablement juste l'année

In [0]:
from pyspark.sql.functions import to_date, col, when, split, size, concat, lit

nb_segments = size(split(col('release_date'), '/'))

clean_df = clean_df.withColumn(
    'release_date',
    when(col('release_date') == '', None)
    .when(nb_segments == 3, to_date(col('release_date'), 'yyyy/M/d'))
    .when(nb_segments == 2, to_date(concat(col('release_date'), lit('/1')), 'yyyy/M/d'))
    .when(nb_segments == 1, to_date(concat(col('release_date'), lit('/1/1')), 'yyyy/M/d'))
    .otherwise(None)
)
clean_df.select('release_date').show(5, truncate=False)

+------------+
|release_date|
+------------+
|2000-11-01  |
|2021-05-14  |
|2020-10-16  |
|2020-10-14  |
|2019-03-30  |
+------------+
only showing top 5 rows


In [0]:
price_cols = [ 'price', 'initialprice']

for c in price_cols:
    clean_df = clean_df.withColumn(c, col(c) / 100)
clean_df.select('price', 'initialprice').show(5)


+-----+------------+
|price|initialprice|
+-----+------------+
| 9.99|        9.99|
| 9.99|        9.99|
| 5.99|       19.99|
|19.99|       19.99|
| 1.99|        1.99|
+-----+------------+
only showing top 5 rows


Then, deal with latest missing values.

In [0]:
from pyspark.sql.functions import col, when, to_date, lit

for field in clean_df.schema.fields:
    c = field.name
    if field.dataType.simpleString() == 'string':
        clean_df = clean_df.withColumn(c, when(col(c) == '', 'None').otherwise(col(c)))
    elif c == 'release_date':
        clean_df = clean_df.withColumn(c, when(col(c).isNull(), to_date(lit('1900-01-01'), 'yyyy-MM-dd')).otherwise(col(c)))

clean_df.filter(col('release_date') == to_date(lit('1900-01-01'), 'yyyy-MM-dd')).show(1)

+------+------+---+--------------------+--------+-----------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+--------+--------------------+
|    id| appid|ccu|           developer|discount|      genre|        header_image|initialprice|           languages|                name|negative|              owners|positive|price|           publisher|release_date|required_age|   short_description|type|             website|            category|platform|                 tag|
+------+------+---+--------------------+--------+-----------+--------------------+------------+--------------------+--------------------+--------+--------------------+--------+-----+--------------------+------------+------------+--------------------+----+--------------------+--------------------+--------+--------------------+
|102500|102500| 

In [0]:
clean_df.filter(col('release_date').isNull()).count()

0

Then, lowercase all values.

In [0]:
from pyspark.sql.functions import lower, col

string_cols = [f.name for f in clean_df.schema.fields if f.dataType.simpleString() == 'string']

clean_df = clean_df.select([lower(col(c)).alias(c) for c in string_cols] + [col(c) for c in clean_df.columns if c not in string_cols])
clean_df.show(1)

+---+-----+---------+------+--------------------+--------------------+--------------+--------+--------------------+--------+---------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
| id|appid|developer| genre|        header_image|           languages|          name|negative|              owners|positive|publisher|required_age|   short_description|type|website|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|
+---+-----+---------+------+--------------------+--------------------+--------------+--------+--------------------+--------+---------+------------+--------------------+----+-------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
| 10|   10|    valve|action|https://cdn.akama...|english, french, ...|counter-strike|    5199|10,000,000 .. 20,...|  201215|   

Endly, drop unused columns.

In [0]:
clean_df = clean_df.drop('id', 'type', 'header_image', 'website')

### b) verify clean_df dataframe

In [0]:
clean_df.createOrReplaceTempView('table') # Refresh the temporary view table in memory
result = spark.sql("select * from table limit 1")
result.show()

+-----+---------+------+--------------------+--------------+--------+--------------------+--------+---------+------------+--------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
|appid|developer| genre|           languages|          name|negative|              owners|positive|publisher|required_age|   short_description|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|
+-----+---------+------+--------------------+--------------+--------+--------------------+--------+---------+------------+--------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
|   10|    valve|action|english, french, ...|counter-strike|    5199|10,000,000 .. 20,...|  201215|    valve|           0|play the world's ...|multi-player, val...|linux, mac, windows|{"1980s":266,"199...|13990|       0|        9.99| 9.99|  

In [0]:
clean_df.printSchema()

root
 |-- appid: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- category: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)
 |-- ccu: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- initialprice: double (nullable = true)
 |-- price: double (nullable = true)
 |-- release_date: date (nullable = true)



In [0]:
# Number of elements in dataframe
print(f"Entries number : {clean_df.count()}")

Entries number : 55691


In [0]:
result = spark.sql("select * from table limit(5)")
result.show()

+-------+--------------------+--------------------+--------------------+--------------------+--------+--------------------+--------+--------------------+------------+-------------------------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
|  appid|           developer|               genre|           languages|                name|negative|              owners|positive|           publisher|required_age|                    short_description|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|
+-------+--------------------+--------------------+--------------------+--------------------+--------+--------------------+--------+--------------------+------------+-------------------------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
|     10|               valve|              action|english

In [0]:
clean_df.select('name', 'release_date', 'developer', 'publisher', 'owners', 'ccu').sample(fraction=0.0001).show(truncate=False)

+----------------------------------------+------------+--------------------------+--------------------------+--------------------+---+
|name                                    |release_date|developer                 |publisher                 |owners              |ccu|
+----------------------------------------+------------+--------------------------+--------------------------+--------------------+---+
|medieval monarch                        |2020-01-01  |mr.lordy                  |mr.lordy                  |20,000 .. 50,000    |0  |
|there you are                           |2021-12-07  |funky dango               |funky dango               |0 .. 20,000         |0  |
|visual novel sisters                    |2021-09-23  |ultimate 3d novels        |ultimate 3d novels        |0 .. 20,000         |0  |
|everquest                               |2012-12-13  |darkpaw games             |daybreak game company     |500,000 .. 1,000,000|461|
|tobari and the night of the curious moon|2015-05-26  |

Verify missing values.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

all_cols = clean_df.columns
string_cols = [f.name for f in clean_df.schema.fields if f.dataType.simpleString() == 'string']

clean_df.select([spark_sum((col(c).isNull() | ((col(c) == '') if c in string_cols else False)).cast('int')).alias(c) for c in all_cols]).show()

+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+
|appid|developer|genre|languages|name|negative|owners|positive|publisher|required_age|short_description|category|platform|tag|ccu|discount|initialprice|price|release_date|
+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+
|    0|        0|    0|        0|   0|       0|     0|       0|        0|           0|                0|       0|       0|  0|  0|       0|           0|    0|           0|
+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+



In [0]:
result = spark.sql("select * from table where developer = 'none' or genre = 'none' or languages = 'none' or publisher = 'none' or short_description = 'none' or category = 'none'")
result.limit(1).show()

+-------+----------------+---------+---------+--------------------+--------+----------------+--------+----------------+------------+--------------------+--------+--------+--------------------+---+--------+------------+-----+------------+
|  appid|       developer|    genre|languages|                name|negative|          owners|positive|       publisher|required_age|   short_description|category|platform|                 tag|ccu|discount|initialprice|price|release_date|
+-------+----------------+---------+---------+--------------------+--------+----------------+--------+----------------+------------+--------------------+--------+--------+--------------------+---+--------+------------+-----+------------+
|1000510|1337 game design|education|  english|the marvellous ma...|      21|20,000 .. 50,000|     104|1337 game design|           0|the marvellous ma...|    none| windows|{"education":21,"...|  0|       0|         0.0|  0.0|  2019-02-11|
+-------+----------------+---------+---------+--

In [0]:
clean_df.filter(col('release_date').isNull()).count()

0

In [0]:
result = clean_df \
    .filter(col('release_date') == to_date(lit('1900-01-01'), 'yyyy-MM-dd'))
result.limit(1).show()

+------+--------------------+-----------+--------------------+--------------------+--------+--------------------+--------+--------------------+------------+--------------------+--------------------+--------+--------------------+---+--------+------------+-----+------------+
| appid|           developer|      genre|           languages|                name|negative|              owners|positive|           publisher|required_age|   short_description|            category|platform|                 tag|ccu|discount|initialprice|price|release_date|
+------+--------------------+-----------+--------------------+--------------------+--------+--------------------+--------+--------------------+------------+--------------------+--------------------+--------+--------------------+---+--------+------------+-----+------------+
|102500|big huge games, 3...|action, rpg|english, french, ...|kingdoms of amalu...|    1492|2,000,000 .. 5,00...|   11184|38 studios, elect...|          15|the minds of new ...|s

In [0]:

clean_df.describe().toPandas()

,summary,appid,developer,genre,languages,name,negative,owners,positive,publisher,required_age,short_description,category,platform,tag,ccu,discount,initialprice,price
0,count,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691,55691
1,mean,1025603.0926720655,67391.3,None,None,Infinity,241.8376937027527,None,1470.8755992889335,2498.75,0.1978882344490734,None,None,None,None,138.9596164550825,2.603777989262179,7.975663033519214,7.732849832104521
2,stddev,522784.9683283426,210681.95381245477,None,None,NaN,5765.4137615598665,None,30982.733479535047,1809.1987867561706,2.296292461481821,None,None,None,None,6002.067909130785,12.887080174743156,11.04762477841335,10.931345827234535
3,min,10,"""nieko""",accounting,arabic,! that bastard is trying to steal our gold !,0,"0 .. 20,000",0,"""revday studio""",0,hungry piggy vs chicken: yes! we are hungry!...,captions available,linux,"{""1980s"":10,""1990's"":10,""2d"":10,""action"":29,""c...",0,0,0.0,0.0
4,max,999990,＼上／,web publishing,turkish,～daydream～蝶が舞う頃に,9972,"500,000 .. 1,000,000",9998,ｌｅｍｏｎ ｂａｌｍ,ma 15+,🚗 take part in a roller coaster of emotions wi...,"vr support, steamvr collectibles",windows,{},874053,90,999.0,999.0


In [0]:
result = clean_df \
    .filter(col('release_date') != to_date(lit('1900-01-01'), 'yyyy-MM-dd')) \
    .select( \
        F.min("release_date").alias("min_release_date"),
        F.max("release_date").alias("max_release_date"))
result.show()

+----------------+----------------+
|min_release_date|max_release_date|
+----------------+----------------+
|      1997-06-30|      2022-11-11|
+----------------+----------------+



# TO DO Nico graph time series

## 3. Analysis at the "macro" level

Which publisher has released the most games on Steam?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .distinct() \
    .groupBy('publisher') \
    .agg(
        count('name').alias('game_nb'),
        countDistinct('release_date').alias('release_nb')
        ) \
    .orderBy(desc('release_nb')) \
    .limit(1)
result.show()

+--------------+-------+----------+
|     publisher|game_nb|release_nb|
+--------------+-------+----------+
|big fish games|    424|       405|
+--------------+-------+----------+



What is Valve's position as publisher ?

In [0]:
from pyspark.sql.functions import count, desc

result = clean_df \
    .select(clean_df['publisher'],'name', 'release_date') \
    .groupBy('publisher') \
    .agg( \
        count('name').alias('game_nb'), \
        countDistinct('release_date').alias('release_nb')) \
    .withColumn('rank', row_number().over(Window
    .orderBy(desc('release_nb')))) \
    .filter(clean_df.publisher.isin(['big fish games','valve']))
result.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------------+-------+----------+----+
|     publisher|game_nb|release_nb|rank|
+--------------+-------+----------+----+
|big fish games|    424|       405|   1|
|         valve|     35|        30| 109|
+--------------+-------+----------+----+



How many games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['name']) \
    .distinct() \
    .count() 
print(f"Games number : {result}")

Games number : 55334


How many developers indicated on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer']) \
    .distinct() \
    .count()
print(f"Developer number : {result}")



Developer number : 34552


Which developer most contribute to games on steam ?

In [0]:
result = clean_df \
    .select(clean_df['developer'],'name') \
    .distinct() \
    .groupBy('developer') \
    .agg(count('name').alias('game_nb')) \
    .orderBy(desc('game_nb')) \
    .limit(5)
result.show(truncate=False)

+------------------------+-------+
|developer               |game_nb|
+------------------------+-------+
|choice of games         |140    |
|none                    |132    |
|creobit                 |122    |
|laush dmitriy sergeevich|108    |
|sokpop collective       |98     |
+------------------------+-------+



What are the best rated games?

In [0]:
result = clean_df \
    .select(clean_df['name'],'positive') \
    .groupBy('name') \
    .agg(F.sum('positive').alias('positive_nb')) \
    .orderBy(desc('positive_nb')) \
    .limit(10)
result.show()

+--------------------+-----------+
|                name|positive_nb|
+--------------------+-----------+
|counter-strike: g...|  5943345.0|
|              dota 2|  1534895.0|
|  grand theft auto v|  1229265.0|
| pubg: battlegrounds|  1185361.0|
|            terraria|  1014711.0|
|tom clancy's rain...|   942910.0|
|         garry's mod|   861240.0|
|     team fortress 2|   846407.0|
|                rust|   732520.0|
|       left 4 dead 2|   643836.0|
+--------------------+-----------+



In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .select(clean_df['name'],'positive', 'negative') \
    .groupBy('name') \
    .agg( \
        F.sum('positive').alias('positive_sum'), \
        F.sum('negative').alias('negative_sum') \
        ) \
    .filter((col('positive_sum') > 0) & (col('negative_sum') > 0)) \
    .withColumn('ratio', col('positive_sum') /  col('negative_sum')) \
    .orderBy(desc('positive_sum')) \
    .limit(10)
result.show(truncate=False)

+--------------------------------+------------+------------+------------------+
|name                            |positive_sum|negative_sum|ratio             |
+--------------------------------+------------+------------+------------------+
|counter-strike: global offensive|5943345.0   |787093.0    |7.551007314256384 |
|dota 2                          |1534895.0   |317916.0    |4.82798915436782  |
|grand theft auto v              |1229265.0   |213379.0    |5.760946484893077 |
|pubg: battlegrounds             |1185361.0   |908515.0    |1.3047236424274777|
|terraria                        |1014711.0   |22380.0     |45.34008042895442 |
|tom clancy's rainbow six siege  |942910.0    |143247.0    |6.582406612354884 |
|garry's mod                     |861240.0    |29998.0     |28.709913994266284|
|team fortress 2                 |846407.0    |57423.0     |14.739860334709089|
|rust                            |732520.0    |112160.0    |6.531027104136947 |
|left 4 dead 2                   |643836

In absolute value, Counter-Strike: Global Offensive has more positive advices (65,4M). Yet Terraria has a better ratio : 45,34 positive advices for 1 negative advice against 7,5 on Counter-Strike. Then, Terraria is proportianaly more liked.

Quels sont les jeux les plus utilisés au moment où le dataset a été construit ?

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .select(clean_df['name'],'ccu') \
    .orderBy(desc('ccu'))
result.show(truncate=False)

+--------------------------------+------+
|name                            |ccu   |
+--------------------------------+------+
|counter-strike: global offensive|874053|
|dota 2                          |852995|
|pubg: battlegrounds             |339287|
|apex legends                    |314468|
|lost ark                        |273088|
|call of duty: modern warfare ii |206441|
|grand theft auto v              |140671|
|new world                       |127379|
|rust                            |121146|
|naraka: bladepoint              |116729|
|wallpaper engine                |109428|
|team fortress 2                 |108900|
|warhammer: vermintide 2         |85953 |
|war thunder                     |71685 |
|ark: survival evolved           |70578 |
|unturned                        |59366 |
|terraria                        |58984 |
|destiny 2                       |57116 |
|the sims 4                      |55459 |
|dayz                            |53885 |
+--------------------------------+

Et quels sont les jeux les plus achetés ?

In [0]:
from pyspark.sql.functions import col, try_divide

result = clean_df \
    .groupBy('owners') \
    .agg( \
        F.count('owners').alias('owners_sum')) \
    .orderBy('owners_sum')
result.show(truncate=False)

+--------------------------+----------+
|owners                    |owners_sum|
+--------------------------+----------+
|200,000,000 .. 500,000,000|1         |
|50,000,000 .. 100,000,000 |4         |
|20,000,000 .. 50,000,000  |21        |
|10,000,000 .. 20,000,000  |41        |
|5,000,000 .. 10,000,000   |97        |
|2,000,000 .. 5,000,000    |335       |
|1,000,000 .. 2,000,000    |526       |
|500,000 .. 1,000,000      |933       |
|200,000 .. 500,000        |2162      |
|100,000 .. 200,000        |2519      |
|50,000 .. 100,000         |3695      |
|20,000 .. 50,000          |7285      |
|0 .. 20,000               |38072     |
+--------------------------+----------+



In [0]:
from pyspark.sql.functions import col, regexp_replace, split, desc

result = clean_df \
    .select('name', 'owners', 'release_date') \
    .withColumn('owners_num', regexp_replace(split(col('owners'), ' ')[0], ',', '').cast('long')) \
    .orderBy(desc('owners_num')) \
    .drop('owners_num') \
    .limit(10)
result.show(truncate=False)

+--------------------------------+--------------------------+------------+
|name                            |owners                    |release_date|
+--------------------------------+--------------------------+------------+
|dota 2                          |200,000,000 .. 500,000,000|2013-07-09  |
|pubg: battlegrounds             |50,000,000 .. 100,000,000 |2017-12-21  |
|team fortress 2                 |50,000,000 .. 100,000,000 |2007-10-10  |
|counter-strike: global offensive|50,000,000 .. 100,000,000 |2012-08-21  |
|new world                       |50,000,000 .. 100,000,000 |2021-09-28  |
|destiny 2                       |20,000,000 .. 50,000,000  |2019-10-01  |
|apex legends                    |20,000,000 .. 50,000,000  |2020-11-04  |
|terraria                        |20,000,000 .. 50,000,000  |2011-05-16  |
|warframe                        |20,000,000 .. 50,000,000  |2013-03-25  |
|fall guys: ultimate knockout    |20,000,000 .. 50,000,000  |2020-08-03  |
+------------------------

Les jeux les plus plébiscités ne sont pas forcément les plus utilisés ou les plus achetés. counter-strike: global offensive était le jeu le plus utilisé avec 874053 utilisateurs. Et dota 2 était le jeu le plus vendu avec 200 000 000 à 500 000 000 utilisateurs. counter-strike: global offensive n'a que 50 000 000 à 100 000 000 d'acheteurs. Avec la release_date, on s'aperçoit que data 2 et conter-strike offensive sont anciens.

In [0]:
result = clean_df.groupBy('owners').agg(F.count('*').alias('count')).orderBy(desc('count'))
result.show(truncate=False)

+--------------------------+-----+
|owners                    |count|
+--------------------------+-----+
|0 .. 20,000               |38072|
|20,000 .. 50,000          |7285 |
|50,000 .. 100,000         |3695 |
|100,000 .. 200,000        |2519 |
|200,000 .. 500,000        |2162 |
|500,000 .. 1,000,000      |933  |
|1,000,000 .. 2,000,000    |526  |
|2,000,000 .. 5,000,000    |335  |
|5,000,000 .. 10,000,000   |97   |
|10,000,000 .. 20,000,000  |41   |
|20,000,000 .. 50,000,000  |21   |
|50,000,000 .. 100,000,000 |4    |
|200,000,000 .. 500,000,000|1    |
+--------------------------+-----+



Are there years with more releases? Were there more or fewer game releases during the Covid, for example?

In [0]:
from pyspark.sql.functions import (
    col, year, try_to_date, count, countDistinct
)

result = (flat_df \
    .select("name", "release_date") \
    .distinct() \
    .withColumn( \
        "release_year", \
        year(try_to_date(col("release_date"), "yyyy/M/d")) \
    ) \
    .filter(col("release_year").isNotNull()) \
    .groupBy("release_year") \
    .agg( \
        count("*").alias("release_nb"), \
        countDistinct("name").alias("name_nb") \
    ) \
    .filter((col('release_nb') > 0) & (col('name_nb') > 0)) \
    .withColumn('ratio', col('release_nb') /  col('name_nb')) \
    .orderBy(desc("release_year")) \
)

result.show()

+------------+----------+-------+------------------+
|release_year|release_nb|name_nb|             ratio|
+------------+----------+-------+------------------+
|        2022|      7451|   7444|1.0009403546480387|
|        2021|      8805|   8796|1.0010231923601638|
|        2020|      8287|   8278|1.0010872191350568|
|        2019|      6949|   6945| 1.000575953923686|
|        2018|      7663|   7654|1.0011758557616932|
|        2017|      6006|   6003|1.0004997501249375|
|        2016|      4176|   4175|1.0002395209580839|
|        2015|      2566|   2565|1.0003898635477584|
|        2014|      1550|   1550|               1.0|
|        2013|       469|    469|               1.0|
|        2012|       344|    344|               1.0|
|        2011|       267|    267|               1.0|
|        2010|       281|    281|               1.0|
|        2009|       309|    309|               1.0|
|        2008|       159|    159|               1.0|
|        2007|        98|     98|             

Yes. There are years with more releases. 
- In absolute value, 2014 to 2022. There are more releases after the covid's year in 2021 with 8676. 
- In prortional value, 2015 to 2022. Releases become higher than games since 2015. Yet, releases are higher on 2020, the covid's year with 1.0011 releases for one game.  
We observe that Covid accentuated the trend which returns then at normal pace in 2022 with 7401 and 1.00094.

In [0]:
result = spark.sql("SELECT * FROM table") # filters elements from my_table where position
# show everything
result.show()

+-------+----------------------------+--------------------+--------------------+------------------------------------+--------+--------------------+--------+----------------------------+------------+-------------------------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
|  appid|                   developer|               genre|           languages|                                name|negative|              owners|positive|                   publisher|required_age|                    short_description|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|
+-------+----------------------------+--------------------+--------------------+------------------------------------+--------+--------------------+--------+----------------------------+------------+-------------------------------------+--------------------+-------------------+--------------------+-----+----

# TO DO Nico graph avec quartiles ? time series ?
How are the prizes distributed? Are there many games with a discount? regarder avec et sans mettre un count

In [0]:
result = clean_df \
    .select(clean_df['name'], 'price') \
    .orderBy('price')
display(result)

name,price
chronicles of a dark lord: episode ii war of the abyss,0.0
steamdolls - order of chaos : concept demo,0.0
commands & colors: ancients,0.0
digging dragon,0.0
playcraft,0.0
12 hours to die,0.0
neon boost,0.0
sequence storm,0.0
in - sight,0.0
path of exile,0.0


Databricks visualization. Run in Databricks to view.

In [0]:
result = clean_df \
    .groupBy('price') \
    .agg( \
        F.count('price').alias('game_nb')) \
    .orderBy('price')
display(result)

price,game_nb
0.0,7780
0.28,20
0.29,11
0.3,2
0.31,6
0.37,19
0.38,7
0.39,7
0.41,10
0.44,1


Databricks visualization. Run in Databricks to view.

Prices varies on a scale from 0 to 999 $, sometime of any cents. Mean price is 7,73 $ with an initial price higher on 7,93 $ (cf last run clean_df.describe() command) (cf last run clean_df.describe() command). 

In [0]:
result = clean_df \
    .groupBy(
        (F.floor(F.col('price') / 10) * 10).alias('range_price')) \
    .agg( \
        F.count('price').alias('game_nb')) \
    .orderBy('range_price')
display(result)

range_price,game_nb
0,43708
10,9022
20,1794
30,600
40,239
50,206
60,34
70,16
80,6
90,29


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Prices don't change ? 

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'name', 'release_date', 'price', 'initialprice', 'discount') \
    .filter((clean_df['price'] != clean_df['initialprice']) & (clean_df['discount'] == 0)) \
    .orderBy('publisher', 'name', 'release_date') \
    .show(truncate=False)

+---------+----+------------+-----+------------+--------+
|publisher|name|release_date|price|initialprice|discount|
+---------+----+------------+-----+------------+--------+
+---------+----+------------+-----+------------+--------+



Prices change only with discount.

Prices varies relying to discount. Le discount varie de 0 à 90 avec une valeur moyenne de 2,6. Le discount est rare.

In [0]:
result = clean_df \
    .select(clean_df['publisher'], 'developer', 'name', 'owners', 'positive', 'negative', 'release_date', 'price', 'initialprice', 'discount', 'genre', 'languages', 'required_age', 'category', 'platform', ) \
    .filter(clean_df.discount != 0 ) \
    .orderBy(desc('discount')) \
    .show()

+--------------------+--------------------+--------------------+------------------+--------+--------+------------+-----+------------+--------+--------------------+--------------------+------------+--------------------+-------------------+
|           publisher|           developer|                name|            owners|positive|negative|release_date|price|initialprice|discount|               genre|           languages|required_age|            category|           platform|
+--------------------+--------------------+--------------------+------------------+--------+--------+------------+-----+------------+--------+--------------------+--------------------+------------+--------------------+-------------------+
|        game dynasty|        game dynasty|            obsurity|  20,000 .. 50,000|     156|      46|  2020-02-24| 0.69|        6.99|      90|action, adventure...|    english, russian|           0|single-player, st...|            windows|
|mass creation, na...|       mass creation| 

Vérifier la répartition des jeux discount

What are the most represented languages?

In [0]:
result = clean_df \
    .select(clean_df['languages'], 'name') \
    .filter(clean_df['name'] == 'counter-strike')
result.show(truncate=False)

+--------------------------------------------------------------------------------------------------+--------------+
|languages                                                                                         |name          |
+--------------------------------------------------------------------------------------------------+--------------+
|english, french, german, italian, spanish - spain, simplified chinese, traditional chinese, korean|counter-strike|
+--------------------------------------------------------------------------------------------------+--------------+



In [0]:
clean_df.select(explode(split(trim(col("languages")), ","))).show()

+--------------------+
|                 col|
+--------------------+
|             english|
|              french|
|              german|
|             italian|
|     spanish - spain|
|  simplified chinese|
| traditional chinese|
|              korean|
|             english|
|              korean|
|  simplified chinese|
|  simplified chinese|
|             english|
|            japanese|
| traditional chinese|
|              french|
|              german|
|     spanish - spain|
|             russian|
| portuguese - brazil|
+--------------------+
only showing top 20 rows


In [0]:
clean_df.select(explode(split(trim(col("languages")), ","))).groupBy("col").count().orderBy("count", ascending=False).show()

+--------------------+-----+
|                 col|count|
+--------------------+-----+
|             english|54646|
|              german|13996|
|              french|13406|
|             russian|12839|
|     spanish - spain|12224|
|  simplified chinese|12213|
|            japanese|10170|
|             italian| 9297|
| portuguese - brazil| 6739|
|              korean| 6575|
| traditional chinese| 6263|
|              polish| 5369|
| portuguese - por...| 4011|
|             turkish| 3601|
|               dutch| 3076|
| spanish - latin ...| 2729|
|               czech| 2340|
|             swedish| 2047|
|           ukrainian| 1928|
|           hungarian| 1923|
+--------------------+-----+
only showing top 20 rows


In [0]:
result = clean_df \
    .filter(clean_df['name'] == 'counter-strike') \
    .select(explode(split(trim(col("languages")), ",")).alias('language')) \
    .groupBy("language") \
    .count() \
    .orderBy("count", ascending=False)
result.show()

+--------------------+-----+
|            language|count|
+--------------------+-----+
|              german|    1|
|              french|    1|
|             italian|    1|
|     spanish - spain|    1|
|              korean|    1|
|  simplified chinese|    1|
|             english|    1|
| traditional chinese|    1|
+--------------------+-----+



# clean languages

## 1)

In [0]:
clean_df.filter((col("languages").like("%#lang_%")) | (col("languages").like("%(all with full audio support)%"))).show(truncate=False)

+-----+-------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------+--------+----------------------+--------+--------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
result = clean_df \
    .filter(
        clean_df['languages'].like("%#lang_%") 
        | clean_df['languages'].like("%(all with full audio support)%")
        | clean_df['languages'].like("%(full audio)%")
        | clean_df['languages'].like("%(not supported)%")
        | clean_df['languages'].like("%(text only)%")
        | clean_df['languages'].like("%;%")
        | clean_df['languages'].like("%[b]*[/b]%")
        )
result.show(truncate=False)

+------+---------------+--------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------+--------+----------------------+--------+---------------------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------+------------+----------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
result = clean_df.filter(
    clean_df["languages"].rlike(r"#lang_|\(all with full audio support\)|\(full audio\)|\(not supported\)|\(text only\)|;|\[b\]\*\[/b\]|\"")
)
result.show(truncate=False)

+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+
|appid|developer|genre|languages|name|negative|owners|positive|publisher|required_age|short_description|category|platform|tag|ccu|discount|initialprice|price|release_date|
+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+
+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+



In [0]:
clean_df = clean_df \
    .withColumn("languages", regexp_replace(clean_df["languages"], r'#lang_|\(all with full audio support\)|\(text only\)|\(full audio\)|\(not supported\)|;|\[b\]\*\[\/b\]|,\s*,', ''))
clean_df.show()

+-------+----------------------------+--------------------+--------------------+------------------------------------+--------+--------------------+--------+----------------------------+------------+-------------------------------------+--------------------+-------------------+--------------------+-----+--------+------------+-----+------------+
|  appid|                   developer|               genre|           languages|                                name|negative|              owners|positive|                   publisher|required_age|                    short_description|            category|           platform|                 tag|  ccu|discount|initialprice|price|release_date|
+-------+----------------------------+--------------------+--------------------+------------------------------------+--------+--------------------+--------+----------------------------+------------+-------------------------------------+--------------------+-------------------+--------------------+-----+----

In [0]:
result = clean_df \
    .filter(
        clean_df['languages'].like("%#lang_%") 
        | clean_df['languages'].like("%(all with full audio support)%")
        | clean_df['languages'].like("%(full audio)%")
        | clean_df['languages'].like("%(not supported)%")
        | clean_df['languages'].like("%(text only)%")
        | clean_df['languages'].like("%;%")
        | clean_df['languages'].like("%[b]*[/b]%")
        )
result.show(truncate=False)

+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+
|appid|developer|genre|languages|name|negative|owners|positive|publisher|required_age|short_description|category|platform|tag|ccu|discount|initialprice|price|release_date|
+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+
+-----+---------+-----+---------+----+--------+------+--------+---------+------------+-----------------+--------+--------+---+---+--------+------------+-----+------------+



In [0]:
result = clean_df \
    .select(explode(split(col("languages"), r",\s*")).alias('language')) \
    .groupBy("language") \
    .count() \
    .orderBy("language")
result.display()

language,count
,1
afrikaans,4
albanian,4
arabic,1796
azerbaijani,2
bangla,3
basque,17
belarusian,23
bosnian,3
bulgarian,1256


In [0]:
result = clean_df \
    .filter( \
        (clean_df['name'] == 'trackmania nations forever') \
        | (clean_df['name'] == 'ninja reflex: steamworks edition') \
        | (clean_df['name'] == 'trackmania united forever'))
result.show(truncate=False)

In [0]:
from pyspark.sql.functions import col
result = clean_df.filter(col("appid").isin([11020, 13000, 7200]))
result.show(truncate=False)

+-----+-------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------+--------+----------------------+--------+--------------+------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------+--------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
clean_df.printSchema()

root
 |-- appid: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: string (nullable = true)
 |-- owners: string (nullable = true)
 |-- positive: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- required_age: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- category: string (nullable = false)
 |-- platform: string (nullable = false)
 |-- tag: string (nullable = true)
 |-- ccu: long (nullable = true)
 |-- discount: long (nullable = true)
 |-- initialprice: double (nullable = true)
 |-- price: double (nullable = true)
 |-- release_date: date (nullable = true)



Are there many games prohibited for children under 16/18?